In [1]:
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, matthews_corrcoef, classification_report, confusion_matrix

In [2]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

Using device: mps


In [3]:
train_df = pd.read_csv("../data/processed_data/train.csv")
dev_df = pd.read_csv("../data/processed_data/dev.csv")
train_df.head()

,Unnamed: 0.1,Unnamed: 0,PID,Text_data,Label
0,2782,9097,dev_pid_207,Finally Leaving : I finally left my husband. W...,moderate
1,3794,10110,dev_pid_1220,Hate... : I hate myself! I hate my stupid fat ...,moderate
2,5080,11495,dev_pid_2605,"I'm just a burden. : I'm annoying, stupid, ugl...",not depression
3,6210,12711,dev_pid_3821,Self-Image : I wish sometimes that I wasn’t se...,not depression
4,3872,10188,dev_pid_1298,I find it kind of funny seeing people freak ou...,moderate


In [4]:
label2id = {
    "not depression": 0,
    "moderate": 1,
    "severe": 2
}
id2label = {v: k for k, v in label2id.items()}
train_df["label"] = train_df["Label"].map(label2id)
dev_df["label"] = dev_df["Label"].map(label2id)

print(train_df["label"].value_counts())
print("\nUnmapped training labels:", train_df["label"].isna().sum())
print("Unmapped dev labels:", dev_df["label"].isna().sum())

label
1    2807
0    2116
2     565
Name: count, dtype: int64

Unmapped training labels: 0
Unmapped dev labels: 0


In [5]:
model_name = "FacebookAI/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [6]:
MAX_LENGTH = 256

train_encodings = tokenizer(
    train_df["Text_data"].astype(str).tolist(),
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH
)

dev_encodings = tokenizer(
    dev_df["Text_data"].astype(str).tolist(),
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH
)

In [7]:
class DepressionDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item["labels"] = torch.tensor(int(self.labels[idx]))
        return item


train_dataset = DepressionDataset(train_encodings, train_df["label"].tolist())
dev_dataset = DepressionDataset(dev_encodings, dev_df["label"].tolist())

print("Training samples:", len(train_dataset))
print("Development samples:", len(dev_dataset))

Training samples: 5488
Development samples: 1373


In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    label2id=label2id,
    id2label=id2label
)

model.to(device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
  

In [9]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "weighted_f1": f1_score(labels, predictions, average="weighted"),
        "balanced_accuracy": balanced_accuracy_score(labels, predictions),
        "mcc": matthews_corrcoef(labels, predictions)
    }

In [10]:
training_args = TrainingArguments(
    output_dir="./roberta_finetuned_results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    seed=14
)

In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics
)

In [12]:
trainer.train()

/Users/kavya/SVNIT/sem5/ml_proj/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Balanced Accuracy,Mcc
1,3.085822,0.834660,0.616169,0.556021,0.592885,0.553303,0.328253
2,2.741881,0.806708,0.634377,0.590743,0.629308,0.592441,0.363749
3,2.259713,0.883599,0.637291,0.598984,0.634475,0.593191,0.365157


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/kavya/SVNIT/sem5/ml_proj/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/kavya/SVNIT/sem5/ml_proj/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2058, training_loss=2.7532291055536593, metrics={'train_runtime': 7248.4527, 'train_samples_per_second': 2.271, 'train_steps_per_second': 0.284, 'total_flos': 2165949654736896.0, 'train_loss': 2.7532291055536593, 'epoch': 3.0})

CLASS WEIGHTED FINE TUINING

In [13]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1, 2])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

print("Class weights:", class_weights)

Class weights: tensor([0.8645, 0.6517, 3.2378], device='mps:0')


In [14]:
weighted_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    label2id=label2id,
    id2label=id2label
)

weighted_model.to(device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
  

In [15]:
from transformers import Trainer

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)

        logits = outputs.logits
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [16]:
weighted_training_args = TrainingArguments(
    output_dir="./roberta_weighted_results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    seed=14
)

In [17]:
weighted_trainer = WeightedTrainer(
    model=weighted_model,
    args=weighted_training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics
)

In [18]:
weighted_trainer.train()

/Users/kavya/SVNIT/sem5/ml_proj/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Balanced Accuracy,Mcc
1,3.370142,0.870020,0.605972,0.555475,0.587563,0.569655,0.319146


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/kavya/SVNIT/sem5/ml_proj/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 